# Model Interpretation: Deepfake Image Classification:

### Import Libraries:

In [10]:
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random

from pathlib import Path
from PIL import Image
from torchvision import transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


In [ ]:
# --------------------------------------------------
# 1. Device
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# --------------------------------------------------
# 2. Find images
# --------------------------------------------------

img_dir = Path(
    r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\raw_images"
)

image_files = list(img_dir.glob("*.jpg"))

if len(image_files) == 0:
    raise FileNotFoundError("No JPG images found.")

print("Total images:", len(image_files))


# --------------------------------------------------
# 3. Select random images
# --------------------------------------------------

num_images = 2

selected_images = random.sample(
    image_files,
    min(num_images, len(image_files))
)

print("\nRandomly selected images:")

for image_path in selected_images:
    print("-", image_path.name)


# --------------------------------------------------
# 4. Image preprocessing
# --------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# --------------------------------------------------
# 5. Load trained ResNet50 model
# --------------------------------------------------

resnet_model = models.resnet50(weights=None)

resnet_model.fc = nn.Sequential(
    nn.Linear(resnet_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

model_path = r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\resnet50_model.pth"

resnet_model.load_state_dict( torch.load(  model_path,  map_location=device  ) )

resnet_model = resnet_model.to(device)
resnet_model.eval()

print("ResNet50 model loaded successfully!")


# --------------------------------------------------
# 6. Grad-CAM
# --------------------------------------------------

target_layer = resnet_model.layer4[-1] 
cam = GradCAM( model=resnet_model, target_layers=[target_layer] )


# --------------------------------------------------
# 7. Binary classification target
# --------------------------------------------------

class BinaryOutputTarget:

    def __call__(self, model_output):
        return model_output[0]


targets = [BinaryOutputTarget()]


# --------------------------------------------------
# 8. Predict and generate Grad-CAM
# --------------------------------------------------

results = []

for image_path in selected_images:

    # Open image
    image = Image.open(image_path).convert("RGB")

    # Preprocess
    input_tensor = transform(image).unsqueeze(0).to(device)


    # Prediction
    with torch.no_grad():

        output = resnet_model(input_tensor)

        probability = torch.sigmoid(output).item()


    # Classification
    prediction = (
        "FAKE"
        if probability >= 0.5
        else "REAL"
    )


    # Grad-CAM
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]


    # Prepare image
    rgb_image = np.array(
        image.resize((224, 224))
    ) / 255.0


    # Grad-CAM visualization
    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )


    # Store results
    results.append({
        "prediction": prediction,
        "probability": probability,
        "original": (rgb_image * 255).astype(np.uint8),
        "gradcam": visualization
    })


    # Print result
    print(
        f"{image_path.name} → "
        f"{prediction} → "
        f"FAKE Probability: {probability * 100:.2f}%"
    )


# --------------------------------------------------
# 9. Create Plotly figure
# --------------------------------------------------

fig = make_subplots(
    rows=len(results),
    cols=2,

    subplot_titles=[
        title
        for result in results
        for title in [
            (
                f"Prediction: {result['prediction']} | "
                f"FAKE Probability: "
                f"{result['probability'] * 100:.2f}%"
            ),
            "Grad-CAM"
        ]
    ],

    vertical_spacing=0.03
)


# --------------------------------------------------
# 10. Add images
# --------------------------------------------------

for i, result in enumerate(results):

    row = i + 1

    # Original image
    fig.add_trace(
        go.Image(
            z=result["original"]
        ),
        row=row,
        col=1
    )

    # Grad-CAM
    fig.add_trace(
        go.Image(
            z=result["gradcam"]
        ),
        row=row,
        col=2
    )


# --------------------------------------------------
# 11. Layout
# --------------------------------------------------

fig.update_layout(
    title="ResNet50 - Random Images with Grad-CAM",
    width=1000,
    height=800
)


# Hide axes
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)


# Display
fig.show()

Using device: cuda
Total images: 5557

Randomly selected images:
- 4575.jpg
- 1030.jpg
ResNet50 model loaded successfully!
4575.jpg → FAKE → FAKE Probability: 100.00%
1030.jpg → REAL → FAKE Probability: 0.00%
